In [0]:
BRONZE_RESTAURANTS = "swiggy.bronze.restaurants"
BRONZE_CUSTOMERS = "swiggy.bronze.customers"

SILVER_RESTAURANTS = "swiggy.silver.restaurants"
SILVER_CUSTOMERS = "swiggy.silver.customers"

In [0]:
from pyspark.sql.functions import current_timestamp, lit , to_date , col , to_timestamp
bronze_customers = spark.table(BRONZE_CUSTOMERS)

silver_customers = (bronze_customers
                    .withColumn("signup_date" , to_date(col("signup_date") , "yyyy-MM-dd HH:mm:ss"))
                    .withColumn("last_order_date" , to_date(col("last_order_date") , "yyyy-MM-dd HH:mm:ss"))
                    .withColumn("created_at" , to_timestamp(col("created_at") , "yyyy-MM-dd HH:mm:ss"))
                    .withColumn("scd_start_date" , current_timestamp())
                    .withColumn("scd_end_date" , lit(None).cast("timestamp"))
                    .withColumn("is_current" , lit(True))
                    .withColumn("scd_version" , lit(1))
                   )


silver_customers.printSchema()

bronze_restaurants = spark.table(BRONZE_RESTAURANTS)
silver_restaurants = (bronze_restaurants
                      .withColumn("created_at" , to_timestamp(col("created_at") , "yyyy-MM-dd HH:mm:ss"))
                      .withColumn("scd_start_date" , current_timestamp())
                      .withColumn("scd_end_date" , lit(None).cast("timestamp"))
                      .withColumn("is_current" , lit(True))
                      .withColumn("scd_version" , lit(1))
                      )
silver_restaurants.printSchema()

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col , lit , when

if spark.catalog.tableExists(SILVER_CUSTOMERS):
    #step 1 : Get current silver rows
    current_silver_customers = spark.table(SILVER_CUSTOMERS).filter(col("is_current") == True)

     #Get the changed 

    changed = (
        silver_customers.alias("s")
        .join(current_silver_customers.alias("t"), "customer_id")
        .where(
            (col("s.customer_city") != col("t.customer_city")) |
            (col("s.customer_state") != col("t.customer_state")) |
            (col("s.subscription_tier") != col("t.subscription_tier")) |
            (col("s.preferred_cuisine") != col("t.preferred_cuisine")) |
            (col("s.rfm_segment") != col("t.rfm_segment")) |
            (col("s.customer_email") != col("t.customer_email")) |
            (col("s.customer_phone") != col("t.customer_phone"))
            )
        .select("s.*")
    )
    #step 2 : close the old records
    delta_table = DeltaTable.forName(spark, SILVER_CUSTOMERS)
    (delta_table.alias("t").merge(
        changed.alias("s"),
        "t.customer_id = s.customer_id AND t.is_current = true"

    ).whenMatchedUpdate(set={
        "is_current": "false",
        "scd_end_date": "current_timestamp()"
    }).execute()
    )


    #step 3 : Insert new versions and brand new customers
    #get the current_scd_max_version for changed customers
    current_version = (
        current_silver_customers
        .groupBy("customer_id")
        .agg({"scd_version": "max"})
        .withColumnRenamed("max(scd_version)", "max_version")
    )

    #increment the version for changed customers
    changed = (
        changed
        .join(current_version, "customer_id")
        .withColumn("scd_version" , when(col("max_version").isNotNull(), col("max_version" )+ 1).otherwise(lit(1))
                    )
        .drop("max_version")
    )

    #Brand new customers that exists in incoming but not in silver at all
    new_customers = silver_customers.join(current_silver_customers, "customer_id", "left_anti")
    #Inserting new and changed (both need inserting)
    to_insert = new_customers.union(changed)

    (
        to_insert.write
            .format("delta")
            .mode("append")
            .saveAsTable(SILVER_CUSTOMERS)
    )

else:
    # First run — write everything fresh
    silver_customers.write \
                    .format("delta") \
                    .saveAsTable(SILVER_CUSTOMERS)


if spark.catalog.tableExists(SILVER_RESTAURANTS):
    #step 1 : Get current silver rows
    current_silver_restaurants = spark.table(SILVER_RESTAURANTS).filter(col("is_current") == True)

     #Get the changed 

    changed = (
        silver_restaurants.alias("s")
        .join(current_silver_restaurants.alias("t"), "restaurant_id")
        .where(
            (col("s.restaurant_city") != col("t.restaurant_city")) |
            (col("s.restaurant_state") != col("t.restaurant_state")) |
            (col("s.cuisine_type") != col("t.cuisine_type")) |
            (col("s.avg_rating") != col("t.avg_rating")) |
            (col("s.avg_preparation_time") != col("t.avg_preparation_time")) |
            (col("s.commission_pct") != col("t.commission_pct")) |
            (col("s.is_active") != col("t.is_active"))
            )
        .select("s.*")
    )
    #step 2 : close the old records
    delta_table = DeltaTable.forName(spark, SILVER_RESTAURANTS)
    (delta_table.alias("t").merge(
        changed.alias("s"),
        "t.restaurant_id = s.restaurant_id AND t.is_current = true"

    ).whenMatchedUpdate(set={
        "is_current": "false",
        "scd_end_date": "current_timestamp()"
    }).execute()
    )


    #step 3 : Insert new versions and brand new customers
    #get the current_scd_max_version for changed customers
    current_version = (
        current_silver_restaurants
        .groupBy("restaurant_id")
        .agg({"scd_version": "max"})
        .withColumnRenamed("max(scd_version)", "max_version")
    )

    #increment the version for changed customers
    changed = (
        changed
        .join(current_version, "restaurant_id")
        .withColumn("scd_version" , when(col("max_version").isNotNull(), col("max_version" )+ 1).otherwise(lit(1))
                    )
        .drop("max_version")
    )

    #Brand new restaurants that exists in incoming but not in silver at all
    new_restaurants = silver_restaurants.join(current_silver_restaurants, "restaurant_id", "left_anti")
    #Inserting new and changed
    to_insert = new_restaurants.union(changed)

    (
        to_insert.write
            .format("delta")
            .mode("append")
            .saveAsTable(SILVER_RESTAURANTS)
    )
else:
    # First run — write everything fresh
    silver_restaurants.write \
                    .format("delta") \
                    .saveAsTable(SILVER_RESTAURANTS)





In [0]:
print("=== Silver Layer Complete Status ===")
print(f"orders_clean      : {spark.table('swiggy.silver.orders_clean').count()} rows")
print(f"delivery_clean    : {spark.table('swiggy.silver.delivery_clean').count()} rows")
print(f"customers         : {spark.table('swiggy.silver.customers').count()} rows")
print(f"restaurants       : {spark.table('swiggy.silver.restaurants').count()} rows")

# Verify SCD columns on restaurants
spark.table("swiggy.silver.restaurants") \
     .select("restaurant_id", "avg_rating", "is_current", "scd_version", "scd_start_date", "scd_end_date") \
     .show(5, truncate=False)